In [1]:
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import time

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

In [3]:
def setup_driver():
  """Setup Chrome driver with options"""
  chrome_options = Options()
  chrome_options.page_load_strategy = "eager"

  service = Service(ChromeDriverManager().install())
  driver = webdriver.Chrome(service=service, options=chrome_options)
  driver.set_page_load_timeout(60)

  return driver

## **States - Andhra Pradesh, Bihar, Gujarat**

In [4]:
states = ["ANDHRA PRADESH", "BIHAR", "GUJARAT"]

In [5]:
cols_names = {
    0: "Sno.",
    1: "Candidate",
    2: "Party",
    3: "Criminal Cases",
    4: "Education",
    5: "Age",
    6: "Total Assets",
    7: "Liabilities"
}
ua = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"}

## **2019 Data**

In [6]:
url_19 = "https://www.myneta.info/LokSabha2019/"
html_19 = requests.get(url_19, headers=ua, timeout=30)
print(html_19.status_code)

200


In [7]:
soup_19 = bs(html_19.text, "html.parser")

In [8]:
for card in soup_19.find_all("div", class_="w3-dropdown-click w3-block"):
    print(card.find("button").get_text(strip=True))
    break

ANDAMAN & NICOBAR ISLANDS


In [9]:
url_dict_19 = {}

In [10]:
found = 0
for card in soup_19.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_19[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_19)

{'ANDHRA PRADESH': 'index.php?action=show_constituencies&state_id=34', 'BIHAR': 'index.php?action=show_constituencies&state_id=37', 'GUJARAT': 'index.php?action=show_constituencies&state_id=39'}


In [11]:
data_19 = {}
for state in states:
    data_19[state] = []

In [12]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_19+url_dict_19[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_19[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the ne

In [13]:
print(len(data_19["ANDHRA PRADESH"]))
print(len(data_19["BIHAR"]))
print(len(data_19["GUJARAT"]))

314
619
371


In [14]:
andhra_19 = pd.DataFrame(data_19["ANDHRA PRADESH"])
bihar_19 = pd.DataFrame(data_19["BIHAR"])
gujarat_19 = pd.DataFrame(data_19["GUJARAT"])

In [15]:
andhra_19 = andhra_19.rename(columns=cols_names)

In [16]:
bihar_19 = bihar_19.rename(columns=cols_names)

In [17]:
gujarat_19 = gujarat_19.rename(columns=cols_names)

In [18]:
andhra_19

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,A. Hemanth,IND,0,5th Pass,28,"Rs 50,000\n~ 50 Thou+",Rs 0\n~
1,2,Abdul Waris,SDPI,4,8th Pass,58,"Rs 51,10,000\n~ 51 Lacs+",Rs 0\n~
2,3,Adala Prabhakara Reddy,YSRCP,0,12th Pass,70,"Rs 2,21,16,63,364\n~ 221 Crore+","Rs 15,22,81,255\n~ 15 Crore+"
3,4,Adari Anand Kumar,TDP,0,10th Pass,43,"Rs 41,05,47,212\n~ 41 Crore+","Rs 20,55,91,400\n~ 20 Crore+"
4,5,Adinarayana Inty,BJP,1,Post Graduate,67,"Rs 3,38,85,898\n~ 3 Crore+","Rs 44,79,155\n~ 44 Lacs+"
...,...,...,...,...,...,...,...,...
309,310,Yanamadala Venkata Suresh,IND,0,Graduate,50,"Rs 13,02,000\n~ 13 Lacs+",Rs 0\n~
310,311,Yarlagadda Rama Mohana Rao,Bahujan Maha Party,0,12th Pass,56,"Rs 1,00,86,000\n~ 1 Crore+",Rs 0\n~
311,312,Yeduguri Sandinti Avinash Reddy,YSRCP,4,Post Graduate,34,"Rs 18,69,69,794\n~ 18 Crore+","Rs 2,28,33,319\n~ 2 Crore+"
312,313,Yella Rao Siyyadula,IND,0,12th Pass,37,"Rs 57,000\n~ 57 Thou+",Rs 0\n~


In [ ]:
bihar_19

In [ ]:
gujarat_19

In [ ]:
# andhra_19.to_csv("data/andhra_2019.csv", index=False)
# bihar_19.to_csv("data/bihar_2019.csv", index=False)
# gujarat_19.to_csv("data/gujarat_2019.csv", index=False)

## **2014 Data**

In [ ]:
url_14 = "https://www.myneta.info/ls2014/"
html_14 = requests.get(url_14, headers=ua, timeout=30)
print(html_14.status_code)

In [ ]:
soup_14 = bs(html_14.text, "html.parser")

In [ ]:
url_dict_14 = {}
found = 0
for card in soup_14.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_14[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_14)

In [ ]:
data_14 = {}
for state in states:
    data_14[state] = []

In [ ]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_14+url_dict_14[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_14[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

In [ ]:
print(len(data_14["ANDHRA PRADESH"]))
print(len(data_14["BIHAR"]))
print(len(data_14["GUJARAT"]))

In [ ]:
andhra_14 = pd.DataFrame(data_14["ANDHRA PRADESH"])
bihar_14 = pd.DataFrame(data_14["BIHAR"])
gujarat_14 = pd.DataFrame(data_14["GUJARAT"])

In [ ]:
andhra_14 = andhra_14.rename(columns=cols_names)
bihar_14 = bihar_14.rename(columns=cols_names)
gujarat_14 = gujarat_14.rename(columns=cols_names)

In [ ]:
andhra_14

In [ ]:
bihar_14

In [ ]:
gujarat_14

In [ ]:
# andhra_14.to_csv("data/andhra_2014.csv", index=False)
# bihar_14.to_csv("data/bihar_2014.csv", index=False)
# gujarat_14.to_csv("data/gujarat_2014.csv", index=False)

## **2024 Data**

In [6]:
url_24 = "https://www.myneta.info/LokSabha2024/"
html_24 = requests.get(url_24, headers=ua, timeout=30)
print(html_24.status_code)

200


In [7]:
soup_24 = bs(html_24.text, "html.parser")

In [8]:
url_dict_24 = {}
found = 0
for card in soup_24.find_all("div", class_="w3-dropdown-click w3-block"):
    state = card.find("button").get_text(strip=True)
    if state not in states:
        continue
    found+=1
    url_dict_24[state] = card.find("a", class_="w3-bar-item w3-button w3-padding-small").get("href", "")
    if found == 3:
        break
print(url_dict_24)

{'ANDHRA PRADESH': 'index.php?action=show_constituencies&state_id=2', 'BIHAR': 'index.php?action=show_constituencies&state_id=5', 'GUJARAT': 'index.php?action=show_constituencies&state_id=11'}


In [9]:
data_24 = {}
for state in states:
    data_24[state] = []

In [10]:
driver = setup_driver()
try:
    for state in states:
        state_url = url_24+url_dict_24[state]
        while state_url:
            driver.get(state_url)
            WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, "table.w3-table"))
            )
            first_row = True
            for row in driver.find_elements(By.CSS_SELECTOR, "table.w3-table tr"):
                if first_row:
                    first_row = False
                    continue
                row_dic = {}
                i = 0
                for col in row.find_elements(By.TAG_NAME, "td"):
                    row_dic[i] = col.text.strip()
                    if (i == 6 or i == 7) and row_dic[i] == "":
                        row_dic[i] = col.find_element(By.TAG_NAME, "img").get_attribute("src")
                    i+=1
                data_24[state].append(row_dic)
            next_url = ""
            for page in driver.find_elements(By.CSS_SELECTOR, "center a"):
                if page.text.strip()=="Next":
                    next_url = page.get_attribute("href")
                    break
            if next_url:
                state_url = next_url
            else:
                state_url = ""
            print("Moving to the next page")
            time.sleep(2)
finally:
    driver.quit()

Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the next page
Moving to the ne

In [11]:
print(len(data_24["ANDHRA PRADESH"]))
print(len(data_24["BIHAR"]))
print(len(data_24["GUJARAT"]))

450
496
266


In [12]:
andhra_24 = pd.DataFrame(data_24["ANDHRA PRADESH"])
bihar_24 = pd.DataFrame(data_24["BIHAR"])
gujarat_24 = pd.DataFrame(data_24["GUJARAT"])

In [13]:
andhra_24 = andhra_24.rename(columns=cols_names)
bihar_24 = bihar_24.rename(columns=cols_names)
gujarat_24 = gujarat_24.rename(columns=cols_names)

In [14]:
andhra_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,L. Khaja Hussain,IND,0,Graduate,42,"Rs 92,46,142\n~ 92 Lacs+",Rs 0\n~
1,2,A Chinna Moulali,BSP,0,Graduate Professional,43,"Rs 85,08,000\n~ 85 Lacs+","Rs 13,00,000\n~ 13 Lacs+"
2,3,A. Madhu,Jai Hindustan Party,0,10th Pass,31,"Rs 54,300\n~ 54 Thou+",Rs 0\n~
3,4,A. Purna Chandra,IND,0,10th Pass,37,"Rs 2,00,000\n~ 2 Lacs+",Rs 0\n~
4,5,A. Varaprasad,Jatiya Jana Sena Party,0,12th Pass,51,"Rs 2,53,427\n~ 2 Lacs+","Rs 1,23,000\n~ 1 Lacs+"
...,...,...,...,...,...,...,...,...
445,446,Yeddu Lakshminarayana,IND,0,5th Pass,66,"Rs 8,30,000\n~ 8 Lacs+","Rs 2,30,000\n~ 2 Lacs+"
446,447,Yedida Vara Prasad Kumar,Republican Party of India (A),0,12th Pass,55,"Rs 1,00,000\n~ 1 Lacs+",Rs 0\n~
447,448,Yelampalle Govardhan Reddy,All India Forward Bloc,0,12th Pass,48,"Rs 31,11,906\n~ 31 Lacs+",Rs 0\n~
448,449,Yenugupalli Krishna,Jaibhim Rao Bharat Party,4,Graduate,45,https://myneta.info/image_v2.php?myneta_folder...,https://myneta.info/image_v2.php?myneta_folder...


In [15]:
bihar_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,Abu Bakar Rahmani,Country Citizen Party,0,Post Graduate,35,"Rs 13,58,312\n~ 13 Lacs+",Rs 0\n~
1,2,A. K. Singh (Ashok),IND,0,Graduate Professional,69,"Rs 2,20,62,267\n~ 2 Crore+",Rs 0\n~
2,3,Aakash Kumar Singh,INC,0,12th Pass,33,"Rs 6,59,33,186\n~ 6 Crore+",Rs 0\n~
3,4,Aanand Misra,IND,0,12th Pass,41,"Rs 56,66,811\n~ 56 Lacs+","Rs 1,70,000\n~ 1 Lacs+"
4,5,Abdhesh Kumar Roy,CPI,4,12th Pass,72,"Rs 45,54,459\n~ 45 Lacs+",Rs 0\n~
...,...,...,...,...,...,...,...,...
491,492,Vishnu Singh,Bharat Jago Janta Party,0,10th Pass,50,"Rs 31,28,369\n~ 31 Lacs+",Rs 0\n~
492,493,Vishwanath Tudu,IND,0,12th Pass,31,"Rs 1,00,612\n~ 1 Lacs+",Rs 0\n~
493,494,Vivek Thakur,BJP,0,Post Graduate,54,"Rs 5,05,66,297\n~ 5 Crore+","Rs 1,04,44,564\n~ 1 Crore+"
494,495,Yog Narayan Singh,IND,0,Literate,49,"Rs 26,95,200\n~ 26 Lacs+",Rs 0\n~


In [16]:
gujarat_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities
0,1,Aarab Hasam Sumra,IND,0,5th Pass,62,"Rs 2,49,591\n~ 2 Lacs+",Rs 0\n~
1,2,Abdulhaq Ismail Nedariya,IND,1,5th Pass,46,"Rs 46,25,500\n~ 46 Lacs+",Rs 0\n~
2,3,Abdulkuddus,IND,0,5th Pass,49,"Rs 3,10,000\n~ 3 Lacs+",Rs 0\n~
3,4,Advocate J. P. Maraviya,INC,0,Graduate Professional,43,"Rs 1,02,31,739\n~ 1 Crore+","Rs 1,44,000\n~ 1 Lacs+"
4,5,Advocate Navinkumar Shankerbhai Patel,IND,0,Graduate Professional,53,"Rs 7,45,000\n~ 7 Lacs+",Rs 0\n~
...,...,...,...,...,...,...,...,...
261,262,Yusuf Vali Hasanali,IND,0,8th Pass,49,"Rs 27,31,000\n~ 27 Lacs+","Rs 6,00,000\n~ 6 Lacs+"
262,263,Zala Devrajbhai Babubhai,IND,3,8th Pass,34,"Rs 4,83,000\n~ 4 Lacs+",Rs 0\n~
263,264,Zala Nayan J.,IND,0,8th Pass,39,"Rs 1,61,640\n~ 1 Lacs+",Rs 0\n~
264,265,Zala Sanjaykumar Mahobatsinh,IND,0,Graduate,36,"Rs 4,00,000\n~ 4 Lacs+",Rs 0\n~


In [ ]:
# andhra_24.to_csv("data/andhra_2024.csv", index=False)
# bihar_24.to_csv("data/bihar_2024.csv", index=False)
# gujarat_24.to_csv("data/gujarat_2024.csv", index=False)